In [1]:
%load_ext autoreload
%autoreload 2
%autosave 30

Autosaving every 30 seconds


# Compute coupling for DRN models

We will utilize Enesmble Corpula Coupling (ECC) and a Gaussian Corpula Approach (GCA)

## ECC

Steps:
1. **Univariate postprocessing:** Obtain univariately postprocessed marginal distibutions for each margin. Here we use location and variable as the margins. This is simply the ouput of a model that inherits from `DistibutionRegression`.

2. **Quantization:** Draw samples from each marginal predictive distibution. The samples should have the same size as the raw ensemble $M$. We sample at equidistant quantiles $\frac{1}{M+1}, \dots, \frac{M}{M+1}$. Note that for the wind speed the distribution is a truncated normal. This yields the samples $\tilde{x}^l_1, \dots, \tilde{x}^l_M$, where $l \in L=I \times J$, $I$ is the set of stations and $J$ is the set of features (variables).

3. **Ensemble Reordering:** Reorder based on the rank structure of the original ensemble with possible ties resolved at random. For each margin $l$, the order statistics of the raw ensemble values $x_{(1)}^l, \dots, x_{(M)}^l$ introduce a permutation $\sigma_l$ of the integers ${1, \dots, M}$ defined by $\sigma_l(m)=\operatorname{rk}(x_m^l)$ for $m = 1, \dots, M$. Ties in the rank are resolved at random. The respecitve margin of the ECC postprocessed ensemble is then given by $\hat{x}=\tilde{x}_{\sigma_l(1)}, \dots, \tilde{x}_{\sigma_l(M)}$.

## GCA
First of all some problems with this approach:
- In the common case in which the dimension L (= all station, variable, lead time pairs) of the output quantity is huge and no specific structure can be exploited, parametric methods are bound to fail.

Steps:
1. **Transform to latent Gaussian:** Transform a set of past observarions into latent standard Gaussian observations via $$\tilde{y}^l=\Phi^{-1}(F_\theta^l(y^l))$$ where $F_\theta^d$ is the distribution for each margin.
2. **Sampling:** Calculate $\Sigma$ based on $\tilde{y}^l$ obtained in Step 1. Then sample from $N(0,\Sigma)$ to obtain $Z_1, \dots, Z_M$.
3. **Reverse Transform:** Transform back to original space: $$X_m^l=(F_\theta^l)^{-1}(\Phi(Z_m^l))$$



In [2]:
import importlib
from collections import defaultdict

import hydra
import lightning as L
import numpy as np
import pandas as pd
import torch
import xarray as xr
from einops import rearrange, reduce
from omegaconf import DictConfig
from scipy.stats import multivariate_normal, norm, truncnorm
from tqdm import tqdm

from genpp import BASE_DIR
from genpp.configs import add_y_kwargs, del_key, register_resolvers
from genpp.data import (
    FC_VARS,
    FORECAST_ENS_FLAT_AGG_PATH,
    FORECAST_ENS_PATH,
    OBSERVATIONS_FLAT_PATH,
)
from genpp.eval.utils import (
    compute_scores_per_leadtime,
    log_scores,
    save_predictions_dataarray,
    save_scores_df,
    update_wandb_run,
)
from genpp.models.loss import EnergyScore, EnsembleCRPS, VariogramScore

try:
    register_resolvers()
except Exception:
    pass

best emos model: `feik/genpp/k32mygar`

best drn model: `feik/genpp/hn0gdrqm`

In [3]:
RUN_PATH = "feik/genpp/hn0gdrqm"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

In [4]:
# Load model and dataloader to get predictions
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
# Do not shuffle any dataloader
cfg.data.module.dataloader_config.train.shuffle = False
cfg.data.module.dataloader_config.val.shuffle = False
cfg.data.module.dataloader_config.test.shuffle = False
model_class = cfg.model._target_.split(".")[-1]
# TODO this is a workaround for old configs
add_y_kwargs(
    cfg, y_kwargs={"batch_dims": {}, "input_dims": {"feature": 2, "longitude": 37, "latitude": 31}}
)
del_key(cfg.data.module.dataset_config.train.x_kwargs.batch_dims, "time")
del_key(cfg.data.module.dataset_config.train.x_kwargs.batch_dims, "prediction_timedelta")

datamodule = hydra.utils.instantiate(cfg.data.module)


datamodule.prepare_data()
datamodule.setup(stage="validate")
datamodule.setup(stage="test")

Adding y_kwargs to dataset_config.train
Deleted key 'time' from config.
Deleted key 'prediction_timedelta' from config.
Configuration hash: 34a898cebd0034670ff98a364b6ca9ecef3924b56ed289a881b00e5ca0304f3d
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_34a898cebd0034670ff98a364b6ca9ecef3924b56ed289a881b00e5ca0304f3d.pt.


In [41]:
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)
trainer = L.Trainer(logger=False, accelerator="gpu", devices=[1])

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [6]:
test_scores = trainer.test(model, datamodule.test_dataloader())

You are using a CUDA device ('NVIDIA RTX A5000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 4: 100%|██████████| 23/23 [00:00<00:00, 100.77it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0             DataLoader 1             DataLoader 2
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.4347320795059204        0.497993141412735       0.5731497406959534
     test_loss_var_0        0.48289263248443604      0.5546532273292542       0.6387450695037842
     test_loss_var_1        0.3865715265274048       0.44133302569389343      0.5075545907020569
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 3             Data

In [7]:
predictions = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 114/114 [00:01<00:00, 104.17it/s]


In [8]:
def stack_predictions(predictions) -> list[dict[str, torch.Tensor]]:
    n_dicts = len(predictions[0])
    merged = [defaultdict(list) for _ in range(n_dicts)]

    # accumulate tensors per key
    for batch in predictions:
        for i, d in enumerate(batch):
            for k, v in d.items():
                merged[i][k].append(v)

    # concatenate along first dimension
    return [{k: torch.cat(vs, dim=0) for k, vs in m.items()} for m in merged]


def predictions_to_dataarray(obs_da, preds):
    """
    obs_da: original xarray.DataArray of observations
    preds: list of dicts, each dict corresponds to one feature
           dict values are torch.Tensors with shape (time, longitude, latitude)
    """
    features = obs_da.coords["feature"].values  # ["2m_temperature", "10m_wind_speed"]

    new_data = []
    new_feature_names = []

    for feat_name, pred_dict in zip(features, preds):
        for param_name, tensor in pred_dict.items():
            # ensure numpy
            arr = tensor.detach().cpu().numpy()
            new_data.append(arr)
            new_feature_names.append(f"{feat_name}_{param_name}")

    # stack along new "feature" axis
    data = np.stack(new_data, axis=1)  # (time, new_features, longitude, latitude)
    print(data.shape)

    # build DataArray
    return xr.DataArray(
        data,
        coords={
            "prediction": obs_da.coords["prediction"],
            "feature": new_feature_names,
            "longitude": obs_da.coords["longitude"],
            "latitude": obs_da.coords["latitude"],
            "prediction_time": ("prediction", obs_da.coords["prediction_time"].values),
        },
        dims=("prediction", "feature", "longitude", "latitude"),
    )

In [9]:
# Next step get this in order to a nice datastructure
# To ensure the axes are labled correctly we use the train dataset to get the axis labels

stacked = stack_predictions(predictions)
val_split = hydra.utils.instantiate(cfg.data.splits.val)


x_ds = xr.open_dataset(FORECAST_ENS_FLAT_AGG_PATH)

time_slice = {
    "train": hydra.utils.instantiate(cfg.data.splits.train),
    "val": hydra.utils.instantiate(cfg.data.splits.val),
    "test": hydra.utils.instantiate(cfg.data.splits.test),
}
val_slice = time_slice["val"]

x_ds = x_ds.stack(prediction=("time", "prediction_timedelta"))

# Now lets get the actual times we need for the val set and and get the val data
init_times = datamodule.cache_metadata["feature_metadata"]["time"]["val"]
timedeltas = datamodule.cache_metadata["feature_metadata"]["prediction_timedelta"]["val"]
val_times = init_times + timedeltas

val_prediction = pd.MultiIndex.from_arrays(
    [init_times, timedeltas], names=["time", "prediction_timedelta"]
)

# Now load the y_val data for the val times
y_val = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(time=val_times)
    .to_dataarray("feature")
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", val_prediction),
    )
    .swap_dims({"prediction_time": "prediction"})
)
feature_order = list(cfg.data.y_select_variables)
y_da = y_val.sel(feature=feature_order)

predictions_xr = predictions_to_dataarray(y_da, stacked)

(3620, 4, 37, 31)


## ECC Step 2: Quantization
Draw equally spaced samples from the distributions where for the temperature the distibution is $N(\mu, \sigma)$.
And for the wind speed it is a truncated normal with bounds at 0 and $\infty$.
Since the ensemble has 50 members we will draw 50 samples.

In [10]:
def quantile_samples(pred_da, M=50):
    """
    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']
    M: number of quantile samples

    Returns: xarray.DataArray with dims (time, feature, samples, longitude, latitude)
    """
    prediction = pred_da.coords["prediction"]
    longitude = pred_da.coords["longitude"]
    latitude = pred_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    # quantile positions
    qs = np.linspace(1 / (M + 1), M / (M + 1), M)

    # temperature (normal distribution)
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values  # (time, lon, lat)
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values

    temp_samples = norm.ppf(
        qs[:, None, None, None], loc=mu_temp[None, ...], scale=sigma_temp[None, ...]
    )
    # (M, time, lon, lat)

    # wind speed (truncated normal, bounds [0, ∞))
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values

    # From the Docs:
    # If a_trunc and b_trunc are the abscissae at which we wish to truncate the distribution
    # (as opposed to the number of standard deviations from loc),
    # then we can calculate the distribution parameters a and b as follows:
    # a, b = (a_trunc - loc) / scale, (b_trunc - loc) / scale

    a = (0 - mu_wind) / sigma_wind
    b = np.inf
    wind_samples = truncnorm.ppf(
        qs[:, None, None, None],
        a[None, ...],
        b,
        loc=mu_wind[None, ...],
        scale=sigma_wind[None, ...],
    )
    # (M, time, lon, lat)

    # stack along feature axis
    data = np.stack([temp_samples, wind_samples], axis=1)  # (M, feature, time, lon, lat)

    # reorder to (time, feature, sample, lon, lat)
    data = rearrange(data, "sample feature prediction lon lat -> prediction sample feature lon lat")

    return xr.DataArray(
        data,
        coords={
            "prediction": prediction,
            "sample": np.arange(M),
            "feature": features,
            "longitude": longitude,
            "latitude": latitude,
        },
        dims=("prediction", "sample", "feature", "longitude", "latitude"),
    )

In [11]:
pred_samples = quantile_samples(predictions_xr, M=50)

## ECC Step 3: Ensemble Reordering

Now load the Ensemble predictions and determine the rank ordering and use this to reshuffle the predicted samples

In [12]:
ens = (
    xr.open_dataset(FORECAST_ENS_PATH)[FC_VARS]
    .stack(prediction=("time", "prediction_timedelta"))
    .sel(prediction=val_prediction)
    .to_dataarray("feature")
    .transpose("prediction", "number", "feature", "longitude", "latitude")
)

In [13]:
# Random tie breaking is done by noising the data slightly to avoid ties
# Theoretically ties would still be possible, but insanely unlikely
rng = np.random.default_rng(seed=420)
noise = rng.uniform(low=-1e-8, high=1e-8, size=ens.shape)
ens_noised = ens + noise

ens_ranked = ens_noised.rank(dim="number") - 1  # -1 to get ranks from 0 to 49
ens_ranked = ens_ranked.astype(np.int32)

# This should be true
# Ranks reach from 0 to 49
assert (ens_ranked.sum(dim="number") == 49 * 50 / 2).all()  # n(n+1)/2 for the sum

In [14]:
# Get the same dims so that the reodering works
indexer = ens_ranked.copy()
# indexer = indexer.drop_vars("number", errors="ignore")

# Use advanced indexing
# Note to me: I tested this and it works correctly
# the axis called sample will be reordered and called number in the result
# However the coordinate sample will still exist in the result
pred_samples_reordered = pred_samples.isel(sample=indexer)

# Rename number back to sample
pred_samples_reordered = pred_samples_reordered.drop_vars("sample").rename({"number": "sample"})

In [ ]:
save_predictions_dataarray(
    save_path=MODEL_DIR / "val_predictions_ecc.zarr",
    predictions=pred_samples_reordered,
    overwrite=True,
)

## Now compute the energy scores
For this we need to convert the underlying arrays to tensors and collect the scores afterwards
We can loop over the (batched) days to calculate these scores

In [15]:
SKIP_VARIOGRAM = True

In [16]:
crps = EnsembleCRPS()
es = EnergyScore(clamp=False)
vs = VariogramScore(p=0.5)

In [17]:
pred_samples_reordered = pred_samples_reordered.transpose(
    "prediction", "sample", "feature", "longitude", "latitude"
)  # This might be redundant now (keep for safety)

prediction = y_da.prediction
crpss_list = []
ess_per_var_list = []
ess_complete_list = []
if not SKIP_VARIOGRAM:
    vss_per_var_list = []
    vss_complete_list = []
for p in tqdm(prediction):
    curr_obs = y_da.sel(prediction=p).to_numpy()
    curr_pred = pred_samples_reordered.sel(prediction=p).to_numpy()

    obs_t = torch.tensor(curr_obs[None, ...], dtype=torch.float32)
    pred_t = torch.tensor(curr_pred[None, ...], dtype=torch.float32)

    obs_feature = rearrange(obs_t, "b f lon lat -> b f (lon lat)")
    pred_feature = rearrange(pred_t, "b s f lon lat -> b f s (lon lat)")

    obs_complete = rearrange(obs_t, "b f lon lat -> b (f lon lat)")
    pred_complete = rearrange(pred_t, "b s f lon lat -> b s (f lon lat)")

    crps_per_margin = crps(pred_t, obs_t)
    es_per_var = es(pred_feature, obs_feature)
    es_complete = es(pred_complete, obs_complete)
    if not SKIP_VARIOGRAM:
        vs_per_var = vs(pred_feature, obs_feature)
        vs_complete = vs(pred_complete, obs_complete)

    crpss_list.append(crps_per_margin)
    ess_per_var_list.append(es_per_var)
    ess_complete_list.append(es_complete)
    if not SKIP_VARIOGRAM:
        vss_per_var_list.append(vs_per_var)  # type: ignore
        vss_complete_list.append(vs_complete)  # type: ignore

crpss = torch.cat(crpss_list, dim=0)
ess_per_var = torch.cat(ess_per_var_list, dim=0)
ess_complete = torch.cat(ess_complete_list, dim=0)
if not SKIP_VARIOGRAM:
    vss_per_var = torch.cat(vss_per_var_list, dim=0)  # type: ignore
    vss_complete = torch.cat(vss_complete_list, dim=0)  # type: ignore

100%|██████████| 3620/3620 [00:23<00:00, 155.92it/s]


In [18]:
print(f"Mean crps for t2m and 10m wind speed: {reduce(crpss, 't f lat lon -> f', 'mean')}")
print(f"Mean es for t2m and 10m wind speed: {ess_per_var.mean(dim=0)}")
print(f"Mean es complete: {ess_complete.mean(dim=0)}")
if not SKIP_VARIOGRAM:
    print(f"Mean vs for t2m and 10m wind speed: {vss_per_var.mean(dim=0)}")
    print(f"Mean vs complete: {vss_complete.mean(dim=0)}")

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnsembleCRPS",
    variables=datamodule.y_select_variables,
    scores=reduce(crpss, "t f lat lon -> f", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+ECC",
    metric="EnsembleCRPS",
    variables=["combined"],
    scores=reduce(crpss, "t f lat lon -> 1", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+ECC",
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=ess_per_var.mean(dim=0),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+ECC",
    metric="EnergyScore",
    variables=["combined"],
    scores=ess_complete.mean(dim=0, keepdim=True),
)
if not SKIP_VARIOGRAM:
    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+ECC",
        metric="VariogramScore",
        variables=datamodule.y_select_variables,
        scores=vss_per_var.mean(dim=0),
    )

    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+ECC",
        metric="VariogramScore",
        variables=["combined"],
        scores=vss_complete.mean(dim=0, keepdim=True),
    )

ecc_scores = {
    "ECC": {
        "CRPS_combined": reduce(crpss, "t f lat lon -> 1", "mean").item(),
        "CRPS_2m_temperature": reduce(crpss, "t f lat lon -> f", "mean")[0].item(),
        "CRPS_10m_windspeed": reduce(crpss, "t f lat lon -> f", "mean")[1].item(),
        "EnergyScore_combined": ess_complete.mean(dim=0).item(),
        "EnergyScore_2m_temperature": ess_per_var.mean(dim=0)[0].item(),
        "EnergyScore_10m_windspeed": ess_per_var.mean(dim=0)[1].item(),
    }
}
if not SKIP_VARIOGRAM:
    ecc_scores["ECC"].update(
        {
            "VariogramScore_combined": vss_complete.mean(dim=0).item(),
            "VariogramScore_2m_temperature": vss_per_var.mean(dim=0)[0].item(),
            "VariogramScore_10m_windspeed": vss_per_var.mean(dim=0)[1].item(),
        }
    )

Mean crps for t2m and 10m wind speed: tensor([0.6749, 0.5336])
Mean es for t2m and 10m wind speed: tensor([29.1439, 23.4646])
Mean es complete: 38.042991638183594


In [19]:
# Scores per leadtime
ecc_scores_delta = compute_scores_per_leadtime(
    pred_samples_reordered.prediction_timedelta.values,
    crpss,
    ess_per_var,
    ess_complete,
    vss_per_var if not SKIP_VARIOGRAM else None,
    vss_complete if not SKIP_VARIOGRAM else None,
    method="ECC",
)
ecc_scores_delta

Processing leadtime 24h with 728 samples
Processing leadtime 48h with 726 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 722 samples
Processing leadtime 120h with 720 samples


{'ECC': {'CRPS_combined': {'24h': 0.42727288603782654,
   '48h': 0.499896764755249,
   '72h': 0.5864143967628479,
   '96h': 0.6960852742195129,
   '120h': 0.8141457438468933},
  'CRPS_2m_temperature': {'24h': 0.47362393140792847,
   '48h': 0.554633617401123,
   '72h': 0.6518740653991699,
   '96h': 0.7752712965011597,
   '120h': 0.9220896363258362},
  'CRPS_10m_windspeed': {'24h': 0.38092178106307983,
   '48h': 0.4451599717140198,
   '72h': 0.5209548473358154,
   '96h': 0.6168996095657349,
   '120h': 0.7062018513679504},
  'EnergyScore_combined': {'24h': 27.325393676757812,
   '48h': 31.80629539489746,
   '72h': 37.05805587768555,
   '96h': 43.56894302368164,
   '120h': 50.617454528808594},
  'EnergyScore_2m_temperature': {'24h': 21.073789596557617,
   '48h': 24.38852310180664,
   '72h': 28.345090866088867,
   '96h': 33.18257522583008,
   '120h': 38.85191345214844},
  'EnergyScore_10m_windspeed': {'24h': 16.981786727905273,
   '48h': 19.78940200805664,
   '72h': 22.989656448364258,
   '

# GCA Postprocessing
## Step 1
Transforming to latent distribution to estimate $\Sigma$
What we need for that:
- Observations from the train Set
- Forecasts from the train set

In [20]:
# Now lets get the actual times we need for the val set and and get the val data
init_times = datamodule.cache_metadata["feature_metadata"]["time"]["train"]
timedeltas = datamodule.cache_metadata["feature_metadata"]["prediction_timedelta"]["train"]
train_prediction_times = init_times + timedeltas
train_predictions = pd.MultiIndex.from_arrays(
    [init_times, timedeltas], names=["time", "prediction_timedelta"]
)


y_train = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(time=train_prediction_times)
    .to_dataarray("feature")
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", train_predictions),
    )
    .swap_dims({"prediction_time": "prediction"})
)
feature_order = list(cfg.data.y_select_variables)
y_train = y_train.sel(feature=feature_order)

In [21]:
datamodule.setup(stage="fit")
predictions_train = trainer.predict(model, datamodule.train_dataloader(), return_predictions=True)
predictions_train = stack_predictions(predictions_train)
predictions_train = predictions_to_dataarray(y_train, predictions_train)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 341/341 [00:02<00:00, 114.22it/s]
(10905, 4, 37, 31)


In [22]:
def transform_to_latent_gaussian(obs_da, pred_da, eps=1e-9):
    """
    Transform observations into latent Gaussian space.

    obs_da: xarray.DataArray with dims (time, feature, longitude, latitude)
            containing observed values. Feature names must match:
            ["2m_temperature", "10m_wind_speed"]

    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']

    Returns: xarray.DataArray with dims (time, feature, longitude, latitude)
             containing latent Gaussian transformed observations.
    """

    prediction = obs_da.coords["prediction"]
    longitude = obs_da.coords["longitude"]
    latitude = obs_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    # ---- Temperature (normal) ----
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values
    obs_temp = obs_da.sel(feature="2m_temperature").values

    # Compute CDF then transform
    cdf_temp = norm.cdf(obs_temp, loc=mu_temp, scale=sigma_temp)
    # Add clamping to avoid infs
    cdf_temp = np.clip(cdf_temp, eps, 1 - eps)
    latent_temp = norm.ppf(cdf_temp)

    # ---- Wind speed (truncated normal, [0, inf)) ----
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values
    obs_wind = obs_da.sel(feature="10m_wind_speed").values

    a = (0 - mu_wind) / sigma_wind
    b = np.inf
    cdf_wind = truncnorm.cdf(obs_wind, a=a, b=b, loc=mu_wind, scale=sigma_wind)
    # Add clamping to avoid infs
    cdf_wind = np.clip(cdf_wind, eps, 1 - eps)
    latent_wind = norm.ppf(cdf_wind)

    # ---- Stack back into xarray ----
    data = np.stack([latent_temp, latent_wind], axis=1)  # (time, feature, lon, lat)

    return xr.DataArray(
        data,
        coords={
            "prediction": prediction,
            "feature": features,
            "longitude": longitude,
            "latitude": latitude,
        },
        dims=("prediction", "feature", "longitude", "latitude"),
    )

In [23]:
latent = transform_to_latent_gaussian(y_train, predictions_train, eps=1e-7)

In [24]:
flat = latent.stack(space=("feature", "longitude", "latitude"))
# shape: (time, space)
X = flat.values
# TODO Comput sigma per lead time or overall?
# Lets go for overall for now
# But it would make sense to use diffent dependency structures for different lead times
Sigma = np.cov(X, rowvar=False)  # shape (space, space)

## Step 2: Samples from the latent Space

In [25]:
t_steps = len(predictions_xr.prediction)
n_samples = 50

latent_samples = multivariate_normal.rvs(
    mean=np.zeros(Sigma.shape[0]),
    cov=Sigma,  # type: ignore
    size=(t_steps, n_samples),  # type: ignore
)
latent_samples = latent_samples.reshape(t_steps, n_samples, *y_da.shape[1:])

## Step 3: Transform back to the initial space using the predicted Distributions

In [26]:
def inverse_transform_latent(latent_samples, pred_da, eps=1e-9):
    """
    Transform latent Gaussian samples back to original space.

    latent_samples: np.array of shape (time, n_samples, feature, lon, lat)
    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']

    Returns: xarray.DataArray of shape (time, sample, feature, lon, lat)
    """
    prediction = pred_da.coords["prediction"]
    lon = pred_da.coords["longitude"]
    lat = pred_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    n_time, n_samples, n_feature, n_lon, n_lat = latent_samples.shape

    # Transform to uniform [0,1] via standard normal CDF
    uniform_samples = norm.cdf(latent_samples)

    # Clamp the uniform samples to avoid issues with PPF
    uniform_samples = np.clip(uniform_samples, eps, 1 - eps)

    # Allocate array for original space
    orig_samples = np.zeros_like(uniform_samples)

    # ---- 2m temperature (normal) ----
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values

    orig_samples[:, :, 0, :, :] = norm.ppf(
        uniform_samples[:, :, 0, :, :],
        loc=mu_temp[:, None, :, :],  # shape: (time, 1, lon, lat)
        scale=sigma_temp[:, None, :, :],  # shape: (time, 1, lon, lat)
    )

    # ---- 10m wind speed (truncated normal, [0, ∞)) ----
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values

    a = (0 - mu_wind) / sigma_wind
    b = np.inf

    orig_samples[:, :, 1, :, :] = truncnorm.ppf(
        uniform_samples[:, :, 1, :, :],
        a=a[:, None, :, :],
        b=b,
        loc=mu_wind[:, None, :, :],
        scale=sigma_wind[:, None, :, :],
    )

    return xr.DataArray(
        orig_samples,
        coords={
            "prediction": prediction,
            "sample": np.arange(n_samples),
            "feature": features,
            "longitude": lon,
            "latitude": lat,
        },
        dims=("prediction", "sample", "feature", "longitude", "latitude"),
    )


gca_preds = inverse_transform_latent(latent_samples, predictions_xr, eps=1e-7)

In [ ]:
save_predictions_dataarray(
    predictions=gca_preds, save_path=MODEL_DIR / "val_predictions_gca.zarr", overwrite=True
)

## Lastly compute the energy scores

In [27]:
prediction = y_da.prediction
crpss_list = []
ess_per_var_list = []
ess_complete_list = []
if not SKIP_VARIOGRAM:
    vss_per_var_list = []
    vss_complete_list = []
for p in tqdm(prediction):
    curr_obs = y_da.sel(prediction=p).to_numpy()
    curr_pred = gca_preds.sel(prediction=p).to_numpy()

    obs_t = torch.tensor(curr_obs[None, ...], dtype=torch.float32)
    pred_t = torch.tensor(curr_pred[None, ...], dtype=torch.float32)

    obs_feature = rearrange(obs_t, "b f lon lat -> b f (lon lat)")
    pred_feature = rearrange(pred_t, "b n f lon lat -> b f n (lon lat)")

    obs_complete = rearrange(obs_t, "b f lon lat -> b (f lon lat)")
    pred_complete = rearrange(pred_t, "b n f lon lat -> b n (f lon lat)")

    crps_per_margin = crps(pred_t, obs_t)
    es_per_var = es(pred_feature, obs_feature)
    es_complete = es(pred_complete, obs_complete)

    crpss_list.append(crps_per_margin)
    ess_per_var_list.append(es_per_var)
    ess_complete_list.append(es_complete)
    if not SKIP_VARIOGRAM:
        vs_per_var = vs(pred_feature, obs_feature)
        vs_complete = vs(pred_complete, obs_complete)

        vss_per_var_list.append(vs_per_var)  # type: ignore
        vss_complete_list.append(vs_complete)  # type: ignore

crpss = torch.cat(crpss_list, dim=0)
ess_per_var = torch.cat(ess_per_var_list, dim=0)
ess_complete = torch.cat(ess_complete_list, dim=0)
if not SKIP_VARIOGRAM:
    vss_per_var = torch.cat(vss_per_var_list, dim=0)  # type: ignore
    vss_complete = torch.cat(vss_complete_list, dim=0)  # type: ignore

100%|██████████| 3620/3620 [00:22<00:00, 164.12it/s]


In [28]:
print(f"Mean crps for t2m and 10m wind speed: {reduce(crpss, 't f lat lon -> f', 'mean')}")
print(f"Mean es for t2m and 10m wind speed: {ess_per_var.mean(dim=0)}")
print(f"Mean es complete: {ess_complete.mean(dim=0)}")
if not SKIP_VARIOGRAM:
    print(f"Mean vs for t2m and 10m wind speed: {vss_per_var.mean(dim=0)}")
    print(f"Mean vs complete: {vss_complete.mean(dim=0)}")

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnsembleCRPS",
    variables=datamodule.y_select_variables,
    scores=reduce(crpss, "t f lat lon -> f", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnsembleCRPS",
    variables=["combined"],
    scores=reduce(crpss, "t f lat lon -> 1", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=ess_per_var.mean(dim=0),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnergyScore",
    variables=["combined"],
    scores=ess_complete.mean(dim=0, keepdim=True),
)
if not SKIP_VARIOGRAM:
    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+GCA",
        metric="VariogramScore",
        variables=datamodule.y_select_variables,
        scores=vss_per_var.mean(dim=0),
    )

    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+GCA",
        metric="VariogramScore",
        variables=["combined"],
        scores=vss_complete.mean(dim=0, keepdim=True),
    )

gca_scores = {
    "GCA": {
        "CRPS_combined": reduce(crpss, "t f lat lon -> 1", "mean").item(),
        "CRPS_2m_temperature": reduce(crpss, "t f lat lon -> f", "mean")[0].item(),
        "CRPS_10m_windspeed": reduce(crpss, "t f lat lon -> f", "mean")[1].item(),
        "EnergyScore_combined": ess_complete.mean(dim=0).item(),
        "EnergyScore_2m_temperature": ess_per_var.mean(dim=0)[0].item(),
        "EnergyScore_10m_windspeed": ess_per_var.mean(dim=0)[1].item(),
    }
}
if not SKIP_VARIOGRAM:
    gca_scores["GCA"].update(
        {
            "VariogramScore_combined": vss_complete.mean(dim=0).item(),
            "VariogramScore_2m_temperature": vss_per_var.mean(dim=0)[0].item(),
            "VariogramScore_10m_windspeed": vss_per_var.mean(dim=0)[1].item(),
        }
    )
gca_scores

Mean crps for t2m and 10m wind speed: tensor([0.6848, 0.5429])
Mean es for t2m and 10m wind speed: tensor([29.3870, 23.7220])
Mean es complete: 38.39381790161133


{'GCA': {'CRPS_combined': 0.6138172149658203,
  'CRPS_2m_temperature': 0.6847715377807617,
  'CRPS_10m_windspeed': 0.5428630709648132,
  'EnergyScore_combined': 38.39381790161133,
  'EnergyScore_2m_temperature': 29.387008666992188,
  'EnergyScore_10m_windspeed': 23.721969604492188}}

In [31]:
gca_scores_delta = compute_scores_per_leadtime(
    gca_preds.prediction_timedelta.values,
    crpss,
    ess_per_var,
    ess_complete,
    vss_per_var if not SKIP_VARIOGRAM else None,
    vss_complete if not SKIP_VARIOGRAM else None,
    method="GCA",
)
gca_scores_delta

Processing leadtime 24h with 728 samples
Processing leadtime 48h with 726 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 722 samples
Processing leadtime 120h with 720 samples


{'GCA': {'CRPS_combined': {'24h': 0.43425190448760986,
   '48h': 0.5081785321235657,
   '72h': 0.5953254103660583,
   '96h': 0.7078848481178284,
   '120h': 0.8261622190475464},
  'CRPS_2m_temperature': {'24h': 0.4811558127403259,
   '48h': 0.563779354095459,
   '72h': 0.6612730026245117,
   '96h': 0.787102460861206,
   '120h': 0.9336621165275574},
  'CRPS_10m_windspeed': {'24h': 0.38734740018844604,
   '48h': 0.4525780975818634,
   '72h': 0.5293777585029602,
   '96h': 0.6286677718162537,
   '120h': 0.7186623811721802},
  'EnergyScore_combined': {'24h': 27.464887619018555,
   '48h': 31.985868453979492,
   '72h': 37.34099197387695,
   '96h': 44.1007194519043,
   '120h': 51.241451263427734},
  'EnergyScore_2m_temperature': {'24h': 21.173810958862305,
   '48h': 24.509544372558594,
   '72h': 28.538711547851562,
   '96h': 33.528812408447266,
   '120h': 39.30926513671875},
  'EnergyScore_10m_windspeed': {'24h': 17.09721565246582,
   '48h': 19.929733276367188,
   '72h': 23.19603729248047,
   '

In [35]:
val_scores_delta = ecc_scores_delta | gca_scores_delta
update_wandb_run(RUN_PATH, {"val": val_scores_delta})

In [45]:
# Also log as a table
ecc_full = {"val": val_scores_delta}
records = []
for dataset, pp_dict in ecc_full.items():
    for pp_model, metrics in pp_dict.items():
        for metric_name, horizons in metrics.items():
            for horizon, value in horizons.items():
                records.append(
                    (f"{model.__class__.__name__}+{pp_model}", dataset, metric_name, horizon, value)
                )
df = pd.DataFrame(records, columns=["method", "dataset", "metric", "horizon", "value"])
save_scores_df(df=df, run_path=RUN_PATH)
df

wandb: Currently logged in as: feik to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,69
lr-AdamW,0.0001
train_loss,0.57824
trainer/global_step,23869
val_loss,0.60371
val_loss_var_0,0.67329
val_loss_var_1,0.53413


,method,dataset,metric,horizon,value
0,DRNModel+ECC,val,CRPS_combined,24h,0.427273
1,DRNModel+ECC,val,CRPS_combined,48h,0.499897
2,DRNModel+ECC,val,CRPS_combined,72h,0.586414
3,DRNModel+ECC,val,CRPS_combined,96h,0.696085
4,DRNModel+ECC,val,CRPS_combined,120h,0.814146
5,DRNModel+ECC,val,CRPS_2m_temperature,24h,0.473624
6,DRNModel+ECC,val,CRPS_2m_temperature,48h,0.554634
7,DRNModel+ECC,val,CRPS_2m_temperature,72h,0.651874
8,DRNModel+ECC,val,CRPS_2m_temperature,96h,0.775271
9,DRNModel+ECC,val,CRPS_2m_temperature,120h,0.922090
